In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [9]:
#--- DIRECTORIES-----
BASE_DIR = '/content/drive/MyDrive/xn-sitstayforever'
DATASET_DIR = f'{BASE_DIR}/datasets'
OUTPUTS_DIR = f'{BASE_DIR}/outputs'
CHECKPOINTS_DIR = f'{BASE_DIR}/checkpoints'
IMAGE_DIR = f'{DATASET_DIR}/product_images'

print(DATASET_DIR)

/content/drive/MyDrive/xn-sitstayforever/datasets


# Setup & Imports

In [10]:
# Installs for missing dependencies not natively supported
!pip install open-clip-torch -q

print("Open CLIP install successful!")

Open CLIP install successful!


In [11]:
#------- IMPORTS -------------
# --System Libraries---
import os
import sys
import random
from pathlib import Path

# --Data Handling---
import numpy as np
import pandas as pd
import requests

# --Visualizations---
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# --Preprocessing & Encoding---
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans



# --Image Processing---
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm # wraps any loop and gives a live progress bar when loading the images in a loop

# --Deep Learning---
import torch
import torchvision.transforms as transforms

# --CLIP---
import open_clip

print("All Libraries have been successfully imported!")

All Libraries have been successfully imported!


# Dataset Loading

In [12]:
import os

BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/xn-sitstayforever'
DATASET_DIR = os.path.join(BASE_DIR, 'datasets')
IMAGE_DIR = os.path.join(DATASET_DIR, 'product_images')

print('BASE_DIR exists:   ', os.path.exists(BASE_DIR))
print('DATASET_DIR exists:', os.path.exists(DATASET_DIR))
print('IMAGE_DIR exists:  ', os.path.exists(IMAGE_DIR))
print('=== Contents of datasets ===')
for item in sorted(os.listdir(DATASET_DIR)):
    print(repr(item))

BASE_DIR exists:    False
DATASET_DIR exists: False
IMAGE_DIR exists:   False
=== Contents of datasets ===


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/xn-sitstayforever/datasets'

In [ ]:
# ------ LOADING THE DATASET ------
''' The dataset consists of 54 images,
text, 2 spreadsheets which need to
be imported, analyzed and clean during
this preprocessing stage'''

# load the pet_cv_dataset.xlsx using pandas
df_images = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='images_cv_features')
df_products = pd.read_excel(f'{DATASET_DIR}/pet_cv_dataset_full.xlsx', sheet_name='product_summary')

# load SSF_CV_Dataset.xlsx using pandas
df_ssf = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='images_cv_features')
df_keywords = pd.read_excel(f'{DATASET_DIR}/SSF_CV_Dataset.xlsx', sheet_name='keywords_dataset')

# --Outputs---#
print(f'(df_images:   {df_images.shape}')
print(f'(df_products:   {df_products.shape}')
print(f'(df_ssf:   {df_ssf.shape}')
print(f'(df_keywords:   {df_keywords.shape}')

print('=' * 50)
''' Note:
df_images is outputting 145,44 instead of 54,44 to follow the 54 images. Its reading all 100 metadata rows plus the
54 rows combined. Will resolve this next section'''

In [ ]:
# ------ LOADING THE IMAGES FROM GOOGLE DRIVE ------
''' This section will handle loading the images from the dataset. The images are stored in a folder on
Google drive

- since the imags can come in different channel formats, RGBA, grayscale, palette-based, it would be best to unify the formats
  - this will help our other models like CLIP, GradCAM etc to interpret the the same format instead of mixed formats



Storing the images and filepaths seperately for useablilty for other libaries and models. Better for memory and speed
'''

images = {} # store image objects in empty dictionary, ordered by filename
image_paths = {} # store full image file paths by filename to ensure we have correct location

# check file name, path, extension and sort
'''tqdm adds a real-time progress bar to the loop to track execution speed
    and time for image uploads to the dictionaries '''
for filename in tqdm(sorted(os.listdir(IMAGE_DIR))):
  if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
    filepath = os.path.join(IMAGE_DIR, filename)

    try:
      img = Image.open(filepath).convert('RGB') # read the image from drive and convert pixels to RGB (to standardize color 3 color channels)
      images[filename] = img
      image_paths[filename] = filepath
    except Exception as e:
      print(f"Error loading image {filename}: {e}")
      continue

print(f'\nLoaded {len(images)} images from {IMAGE_DIR}')


In [ ]:
''' Issue 1: After filtering the rows, seems as if too many rows were removed. Specially the ones that are tier 'low'
Validating what happened

 '''

print("Before filtering:")
for tier, count in df_images['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

print("Does low tier rows have brightness_mean values? :")
low_tier_rows = df_images[df_images['performance_tier'] == 'low']
if low_tier_rows['brightness_mean'].notna().any():
    count_with_brightness = low_tier_rows['brightness_mean'].notna().sum()
    print(f"  Yes, {count_with_brightness} rows have brightness_mean values.")
else:
    count_without_brightness = low_tier_rows['brightness_mean'].isna().sum()
    print(f"  No, {count_without_brightness} rows do not have brightness_mean values.")

In [ ]:
# ---- Data Quality -----------------------

''' this section will be used to filter and clean the features and metadata

After troubleshooting and looking through the dataset spreadsheet again, found that the tier breakdown had missing images
and for those images brightness_mean' was NaN

Of the 145 rows in pet_cv_dataset_full
- 54 rows have computed CV features, where brightness_mean is not null = working images
- 91 rows are metadata-only = no downloaded images (Burts Bees and Bio Groom images)

Of the 54 usable image rows:
- high: 42 images (7 ASINs × 6 images)
- medium: 6 images (1 ASIN × 6 images)
- sponsor: 6 images (1 ASIN × 6 images — Sit Stay Forever)
- low: 0 images — no low tier products were downloaded


Conclusion/Impact:
With no low tier images, downstream model training will need to be adjusted.

treating it like a binary task (high v. medium/low)
OR
2-class problem (high v. sponsor benchmark)

will revisit in Notebook 2
'''

# filter to rows with computed CV features only (drop meta-data rows)
df_clean = df_images[df_images['brightness_mean'].notna()].copy()


# From starter template in xlsx. Convert CV Feature columns to numeric
CV_FEATURES = [
    'brightness_mean', 'contrast_std', 'sharpness_laplacian',
    'white_bg_pct', 'white_bg_compliance', 'product_dominance_score',
    'text_density_pct', 'clutter_score', 'color_warmth', 'saturation_mean',
    'symmetry_score', 'dominant_color_1_pct', 'ocr_word_count', 'ocr_keyword_count'

]


# create a new column for cleaned data and convert values of column to numeric data types
df_clean[CV_FEATURES] = df_clean[CV_FEATURES].apply(pd.to_numeric, errors='coerce') # turn any non-numeric string or invalid values to NaN

print(f'Total # of rows before filtering: {df_images.shape[0]}')
print(f'Total # of rows after filtering: {df_clean.shape[0]}')

# counting the number of values in the performance tier column then storing the label and count in a dictionary
print("~" * 50)
print("Total Tiers present:")
for tier, count in df_clean['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

In [ ]:
# ---- Flagging problematic images ------------------
''' checking if images descriptions align with image instead stating Amazon UI text.
some images that were datascrapped, i found some images with incorrect context'''

# creating new column to mark images theat are good to go
df_clean['quality_flag'] = 'ok'

# --- Detection 1: Amazon Banner Screenshots ---------------
# detext is Amazon product images contain any Amazon UI text
# checking OCR for any promotional/navigational langugage not related to the product
banner_terms = [
    'spend', 'meds', 'prime', 'shop amazon',
    'school', 'get back', 'introducing', 'restrictions apply'
]

# checking if text is in banner terms
def is_banner_txt(ocr_text):
  if pd.isna(ocr_text):
    return False
  ocr_lower = str(ocr_text).lower()
  for term in banner_terms:
    if term in ocr_lower:
      return True
  return False

# running text in ocr_text to see if it contains the banner terms
banner_mask = df_clean['ocr_text'].apply(is_banner_txt)
df_clean.loc[banner_mask, 'quality_flag'] = 'banner_screenshot'

# --- Output # of Banner Screenshots detected -----------
print(f'Total # of banner screenshots: {banner_mask.sum()}')
if banner_mask.sum() > 0:
  # looking through flagged rows for verification
  print(df_clean.loc[banner_mask, ['asin', 'brand', 'image_type_label', 'ocr_text']].to_string(index=False))

# ---



In [ ]:
# ---- Flagging problematic images Continued... ------------------

# --- Detection 2: Low resolution images ----------------
''' Finding images below 400px. Upscaling these images to a consistent px could stretch the image and present more noise.
using the cleaned dataframe, checking columns width_px and height_px to see if they are below 400px

 '''
low_resolution_mask = (df_clean['width_px'] < 400) & (df_clean['height_px'] < 400)
df_clean.loc[low_resolution_mask, 'quality_flag'] = 'low_resolution'

# --- Output # of low res images detected -----------
print(f'Total # of low res images : {low_resolution_mask.sum()}')
if low_resolution_mask.sum() > 0:
  # looking through flagged rows for verification
  print(df_clean.loc[low_resolution_mask, ['asin', 'brand', 'width_px', 'height_px']].to_string(index=False))

In [ ]:
#--- Binary Tier Relabeling -------
'''
This cell will handle the relabeling of the tiers do to no 'low' tier images and the six SSF 'sponsor' tier getting
relabeled for training purposes.

Setting up 2 main tiers 'high' versus 'medium' as of now and will address the imbalance in Notebook 2
'''

# preserve the original performance_tier column by making a copy
df_clean['tier_original'] = df_clean['performance_tier'].copy()

# relabel sponsor tier to high
df_clean['performance_tier'] = df_clean['performance_tier'].replace('sponsor', 'high')

#remove low tiers there is limited data and no downloads
df_clean = df_clean[df_clean['performance_tier'].isin(['high', 'medium'])]

def tier_counter():
  # count the number of items in each tier group
  for tier, count in df_clean['performance_tier'].value_counts().items():
    print(f"  {tier}: {count}")

print('=' * 50)
print(f'Total useable images: {len(df_clean)}')

In [ ]:
#------ Data quality summary -----
print('Data Quality Summary')
print('=' * 55)

#total rows in raw datasets
print(f'Total Rows in raw dataset: {df_images.shape[0]}')

#total rows in cleaned dataset
print(f'Total Usable Rows after filtering the dataset: {df_clean.shape[0]}')

# number of metadata rows dropped
print(f'Metadata rows dropped: {df_images.shape[0] - df_clean.shape[0]}')

print('~' * 55)
print("Number of quality flags:")
for flag, count in df_clean['quality_flag'].value_counts().items():
  print(f'  {flag}: {count}')

print('~' * 55)
tier_counter() # Call the function without wrapping it in another print()

In [ ]:
# ============================================================
# SECTION 4 — EXPLORATORY DATA ANALYSIS
# Compare SSF vs competitors on three headline visual features.
#
# Three stages:
#   Stage 1: naive comparison on ALL images (misleading result)
#   Stage 2: diagnostic — trace the distortion to low-res images
#   Stage 3: corrected comparison on CLEAN images only
# Input: df_clean (54 usable rows from the preprocessing step)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# --- Split by ORIGINAL tier (before the sponsor->high relabeling) ---
# tier_original: 'sponsor' = SSF's own products,
#                'high'/'medium' = competitors.
# Using tier_original (not performance_tier) keeps SSF separable.
ssf       = df_clean[df_clean['tier_original'] == 'sponsor'].copy()
high_comp = df_clean[df_clean['tier_original'] == 'high'].copy()
med_comp  = df_clean[df_clean['tier_original'] == 'medium'].copy()

HEADLINE = ['brightness_mean', 'sharpness_laplacian', 'white_bg_compliance']


def plot_headline(groups, colors, suptitle):
    """Plot the three headline features as a 1x3 grouped bar chart.
    groups: dict of {label: dataframe}. white_bg_compliance is a 0/1
    flag, so its mean is shown as a percentage (compliance rate)."""
    labels = list(groups.keys())
    titles = ['Brightness (0-255)', 'Sharpness (Laplacian)',
              'White-bg compliance rate']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    for ax, feat, title in zip(axes, HEADLINE, titles):
        vals = [groups[l][feat].mean() for l in labels]
        bars = ax.bar(labels, vals, color=colors[:len(labels)])
        ax.set_title(title, fontsize=11, fontweight='bold')
        for bar, val in zip(bars, vals):
            txt = f'{val:.0%}' if feat == 'white_bg_compliance' else f'{val:.1f}'
            ax.text(bar.get_x() + bar.get_width()/2, val, txt,
                    ha='center', va='bottom', fontsize=9)
    fig.suptitle(suptitle, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
naive = pd.DataFrame({
    'SSF':    ssf[HEADLINE].mean(),
    'High':   high_comp[HEADLINE].mean(),
    'Medium': med_comp[HEADLINE].mean(),
})
print('STAGE 1 — group means on ALL images (potentially distorted):')
print(naive.round(2), '\n')

plot_headline({'SSF': ssf, 'High': high_comp, 'Medium': med_comp},
              ['#E07B39', '#4C72B0', '#9AA0A6'],
              'BEFORE cleaning — all images (Medium sharpness is distorted)')

In [ ]:
# Laplacian sharpness is inflated on very small (low-res) images,
# so a few low-res images can dominate a small group's mean.
print('STAGE 2 — sharpness mean vs median (median resists outliers):')
for name, grp in [('SSF', ssf), ('High', high_comp), ('Medium', med_comp)]:
    s = grp['sharpness_laplacian']
    print(f'  {name:<7} mean={s.mean():>9.1f}  median={s.median():>9.1f}  '
          f'min={s.min():>7.1f}  max={s.max():>9.1f}')

print('\nquality_flag breakdown by original tier:')
print(pd.crosstab(df_clean['tier_original'], df_clean['quality_flag']), '\n')

In [ ]:
# Keep quality_flag == 'ok' to drop low-res and banner images that
# distort brightness/sharpness. Medium keeps only 1 clean image,
# so the comparison focuses on SSF vs High competitors.
ssf_ok  = ssf[ssf['quality_flag'] == 'ok'].copy()
high_ok = high_comp[high_comp['quality_flag'] == 'ok'].copy()

clean_summary = pd.DataFrame({
    'SSF_mean':    ssf_ok[HEADLINE].mean(),
    'High_mean':   high_ok[HEADLINE].mean(),
    'SSF_median':  ssf_ok[HEADLINE].median(),
    'High_median': high_ok[HEADLINE].median(),
})
print(f'STAGE 3 — clean images only (SSF: {len(ssf_ok)}, High: {len(high_ok)}):')
print(clean_summary.round(2))

plot_headline({'SSF': ssf_ok, 'High': high_ok},
              ['#E07B39', '#4C72B0'],
              'AFTER cleaning — clean images only (SSF vs High)')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# --- Work on clean images only ---
clean = df_clean[df_clean['quality_flag'] == 'ok'].copy()

# Candidate visual features (same family as the template's CV_FEATURES)
FEATURES = [
    'brightness_mean', 'contrast_std', 'sharpness_laplacian',
    'white_bg_pct', 'white_bg_compliance', 'product_dominance_score',
    'text_density_pct', 'clutter_score', 'color_warmth',
    'saturation_mean', 'symmetry_score', 'ocr_keyword_count',
]

# Binary target: 1 = high performer, 0 = everything else.
# (medium/low have too few clean images to model separately.)
clean['is_high'] = (clean['performance_tier'] == 'high').astype(int)

In [ ]:
# METHOD A — point-biserial correlation between each feature and
# is_high. |r| ranks how strongly a feature separates high vs rest.
corr_rank = (clean[FEATURES]
             .corrwith(clean['is_high'])
             .abs()
             .sort_values(ascending=False))
print('=== Feature ranking by |correlation| with high-performance ===')
print(corr_rank.round(3).to_string(), '\n')

In [ ]:
# METHOD B — Random Forest importance (cross-check).
# Small sample, so treat this as corroboration, not ground truth.
X = StandardScaler().fit_transform(clean[FEATURES].fillna(clean[FEATURES].median()))
y = clean['is_high'].values
rf = RandomForestClassifier(n_estimators=300, random_state=42,
                            class_weight='balanced')
rf.fit(X, y)
rf_rank = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('=== Random Forest feature importance (cross-check) ===')
print(rf_rank.round(3).to_string(), '\n')


In [ ]:
top3 = corr_rank.head(3).index.tolist()
print('Top 3 features separating high performers:', top3, '\n')

# CHART 1 — feature importance bar chart (correlation ranking)
fig, ax = plt.subplots(figsize=(9, 5))
corr_rank.sort_values().plot(kind='barh', color='#4C72B0', ax=ax)
# highlight the top 3 in orange
for i, feat in enumerate(corr_rank.sort_values().index):
    if feat in top3:
        ax.patches[i].set_color('#E07B39')
ax.set_title('Which visual features separate high-performing images?',
             fontsize=12, fontweight='bold')
ax.set_xlabel('|correlation| with high performance')
plt.tight_layout()
plt.show()

# CHART 2 — top 3 features: SSF vs High competitors
# Shows where SSF sits on each of the most predictive features.
ssf_ok  = clean[clean['tier_original'] == 'sponsor']
high_ok = clean[clean['tier_original'] == 'high']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, feat in zip(axes, top3):
    ssf_val, high_val = ssf_ok[feat].mean(), high_ok[feat].mean()
    bars = ax.bar(['SSF', 'High'], [ssf_val, high_val],
                  color=['#E07B39', '#4C72B0'])
    ax.set_title(feat, fontsize=11, fontweight='bold')
    for bar, val in zip(bars, [ssf_val, high_val]):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2f}',
                ha='center', va='bottom', fontsize=9)
fig.suptitle('Top 3 predictive features — SSF vs High competitors',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Work on clean images only (consistent with Tasks 1-2)
clean   = df_clean[df_clean['quality_flag'] == 'ok'].copy()
ssf_ok  = clean[clean['tier_original'] == 'sponsor'].copy()
high_ok = clean[clean['tier_original'] == 'high'].copy()


# Define the benchmark and the "good direction" for each feature.
# benchmark = high-competitor mean.
# direction = 'high' means a larger value is better (SSF fails if below);
#             'low'  means a smaller value is better (SSF fails if above).
CHECKS = {
    'white_bg_compliance': 'high',   # want MORE compliant white background
    'white_bg_pct':        'high',   # want MORE white background
    'color_warmth':        'low',    # want LESS warm (competitors are cooler)
    'text_density_pct':    'low',    # want LESS on-image text
    'brightness_mean':     'high',   # want adequate brightness
    'sharpness_laplacian': 'high',   # want sharper images
}

benchmark = {f: high_ok[f].mean() for f in CHECKS}
print('=== High-competitor benchmark (target values) ===')
for f, v in benchmark.items():
    print(f'  {f:<22} {v:>8.2f}   (better = {CHECKS[f]})')


# Build a per-image diagnostic table.
# For each SSF image and each feature, record the value, whether
# it falls short of the benchmark, and by how much (% gap).
records = []
for _, row in ssf_ok.iterrows():
    rec = {'image': row['image_filename']}
    n_fail = 0
    for f, direction in CHECKS.items():
        val, bench = row[f], benchmark[f]
        # a "fail" = on the wrong side of the benchmark
        fail = (val < bench) if direction == 'high' else (val > bench)
        gap_pct = (val - bench) / bench * 100 if bench != 0 else np.nan
        rec[f] = round(val, 2)
        rec[f + '_fail'] = 'FAIL' if fail else 'ok'
        if fail:
            n_fail += 1
    rec['num_issues'] = n_fail
    records.append(rec)

diag = pd.DataFrame(records).sort_values('num_issues', ascending=False)

# Table 1: values with pass/fail per feature
show_cols = ['image'] + [c for f in CHECKS for c in (f, f + '_fail')] + ['num_issues']
print('\n=== Per-image diagnostic (FAIL = below competitor benchmark) ===')
print(diag[show_cols].to_string(index=False))

# Table 2: plain-language issue list per image (actionable)
DIRECTION_MSG = {
    'white_bg_compliance': 'background not compliant (needs cleaner white bg)',
    'white_bg_pct':        'too little white background',
    'color_warmth':        'colour too warm (shift toward neutral/cool)',
    'text_density_pct':    'too much on-image text (reduce text)',
    'brightness_mean':     'too dark',
    'sharpness_laplacian': 'not sharp enough',
}
print('\n=== Actionable issues per SSF image ===')
for _, row in diag.iterrows():
    issues = [DIRECTION_MSG[f] for f in CHECKS if row[f + '_fail'] == 'FAIL']
    print(f'\n{row["image"]}  ({row["num_issues"]} issue(s)):')
    if issues:
        for msg in issues:
            print(f'   - {msg}')
    else:
        print('   - meets all benchmarks')


# CHART — heatmap-style view: which image fails which feature
# (red = fails benchmark, green = meets it)
fail_matrix = diag.set_index('image')[[f + '_fail' for f in CHECKS]]
fail_matrix = (fail_matrix == 'FAIL').astype(int)   # 1 = fail, 0 = ok
fail_matrix.columns = list(CHECKS.keys())

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(fail_matrix.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(CHECKS)))
ax.set_xticklabels(list(CHECKS.keys()), rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(fail_matrix)))
ax.set_yticklabels(fail_matrix.index, fontsize=9)
ax.set_title('SSF images vs competitor benchmark  '
             '(red = below benchmark, green = meets it)',
             fontsize=11, fontweight='bold')
# annotate cells
for i in range(fail_matrix.shape[0]):
    for j in range(fail_matrix.shape[1]):
        txt = 'X' if fail_matrix.values[i, j] == 1 else ''
        ax.text(j, i, txt, ha='center', va='center', fontsize=9, color='black')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Section 5.1: Setup ----
# IMG_SIZE = 224 standardizes every image to 224x224, the common input size for
# ResNet50 / CLIP. Augmented images are written to a subfolder under OUTPUTS_DIR.

IMG_SIZE = 224
AUG_DIR = os.path.join(OUTPUTS_DIR, 'augmented')
os.makedirs(AUG_DIR, exist_ok=True)

print(f'Target image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Augmented images will be saved to: {AUG_DIR}')

In [ ]:
# ---- Section 5.2: Select which images to augment ----
''' Decide who gets augmented and by how much.
1. Exclude sponsor images (held-out evaluation set per instructor feedback).
2. Compare tier counts and treat the smaller tier as the minority.
3. Compute variations dynamically with len()/counts so nothing is hard-coded;
   if the cleaned dataset changes upstream, just re-run this cell.
'''

# training pool = everything except sponsor images
train_pool = df_clean[df_clean['tier_original'] != 'sponsor'].copy()

tier_counts = train_pool['performance_tier'].value_counts()
print('Training pool tier counts (sponsor excluded):')
for tier, count in tier_counts.items():
    print(f'  {tier}: {count}')

# identify majority / minority dynamically
majority_count = tier_counts.max()
minority_tier = tier_counts.idxmin()
minority_count = tier_counts.min()

# variations per minority image needed to roughly match the majority tier
variations_per_image = max(1, round(majority_count / minority_count))

print(f'\nMinority tier: {minority_tier} ({minority_count} images)')
print(f'Majority tier count: {majority_count}')
print(f'Variations to generate per minority image: {variations_per_image}')

In [ ]:
# ---- Section 5.3: Define the transforms ----
''' Two transform pipelines:
- base_transform: resize only (standardization applied to every image)
- augment_transform: mild random changes to generate new minority-tier variations.
  Keeping the changes random and mild means each pass produces a slightly
  different image without distorting it beyond recognition.
'''

# standardization only (resize to square, no distortion)
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

# augmentation (for generating new minority-tier variations)
augment_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),  # slight crop + resize
    transforms.RandomHorizontalFlip(p=0.5),                     # 50% chance mirror
    transforms.RandomRotation(15),                              # up to +/-15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2),       # mild lighting changes
])

In [ ]:
# ---- Section 5.4: Generate & save images + labels ----
''' A) Standardize every training image and save it.
    B) For the minority tier, generate variations_per_image augmented copies each.
    C) Record a labels row for every saved image and write a labels CSV that
       Notebook 2 can load directly. '''

labels_records = []

def save_image(img, filename, tier, is_augmented):
    """Save one PIL image to AUG_DIR and record its label."""
    out_path = os.path.join(AUG_DIR, filename)
    img.convert('RGB').save(out_path)
    labels_records.append({
        'filename': filename,
        'performance_tier': tier,
        'is_augmented': is_augmented,
    })

# --- A) Standardize every training image (originals) ---
for _, row in train_pool.iterrows():
    fname = row['image_filename']
    tier = row['performance_tier']
    src_path = image_paths.get(fname, os.path.join(IMAGE_DIR, fname))
    try:
        img = Image.open(src_path).convert('RGB')
    except Exception as e:
        print(f'Skip (cannot open) {fname}: {e}')
        continue
    standardized = base_transform(img)
    save_image(standardized, fname, tier, is_augmented=False)

# --- B) Augment the minority tier only ---
minority_rows = train_pool[train_pool['performance_tier'] == minority_tier]
for _, row in minority_rows.iterrows():
    fname = row['image_filename']
    tier = row['performance_tier']
    src_path = image_paths.get(fname, os.path.join(IMAGE_DIR, fname))
    try:
        img = Image.open(src_path).convert('RGB')
    except Exception as e:
        print(f'Skip (cannot open) {fname}: {e}')
        continue
    stem, ext = os.path.splitext(fname)
    for i in range(variations_per_image):
        aug_img = augment_transform(img)
        aug_name = f'{stem}_aug{i}{ext}'
        save_image(aug_img, aug_name, tier, is_augmented=True)

# --- C) Save the labels file ---
labels_df = pd.DataFrame(labels_records)
labels_path = os.path.join(OUTPUTS_DIR, 'augmented_labels.csv')
labels_df.to_csv(labels_path, index=False)

print(f'Total images written: {len(labels_df)}')
print(f'Labels file saved to: {labels_path}')

In [ ]:
# ---- Section 5.5: Verify (before/after counts + preview) ----
''' Confirm the minority tier was balanced up, and preview a few augmented images. '''

print('=== BEFORE augmentation (training pool) ===')
for tier, count in tier_counts.items():
    print(f'  {tier}: {count}')

print('\n=== AFTER augmentation ===')
after_counts = labels_df['performance_tier'].value_counts()
for tier, count in after_counts.items():
    print(f'  {tier}: {count}')

print(f'\nTotal before: {len(train_pool)}  ->  Total after: {len(labels_df)}')

# preview a few augmented images
aug_samples = labels_df[labels_df['is_augmented']].head(4)
if len(aug_samples) > 0:
    fig, axes = plt.subplots(1, len(aug_samples), figsize=(14, 4))
    if len(aug_samples) == 1:
        axes = [axes]
    for ax, (_, r) in zip(axes, aug_samples.iterrows()):
        img = Image.open(os.path.join(AUG_DIR, r['filename']))
        ax.imshow(img)
        ax.set_title(f"{r['performance_tier']} (aug)", fontsize=10)
        ax.axis('off')
    fig.suptitle('Sample augmented images', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()